#### Extract digits from right to left

In [2]:
from numpy.ma.core import minimum

num = 1234
temp = num
digits = []

while temp > 0:
    digit = temp % 10
    digits.insert(0, digit)
    temp = temp // 10

print(f"digits: {digits}")

digits: [1, 2, 3, 4]


In [3]:
def extract_digits(n):
    if n == 0:
        return [0]

    digits = []
    # Handle negative numbers
    sign = -1 if n < 0 else 1
    n = abs(n)

    while n > 0:
        digits.append(n % 10)
        n = n // 10

    # Reverse to maintain original order
    return [d * sign if i == 0 and sign == -1 else d for i, d in enumerate(reversed(digits))]

# Example
num = 12345
print(extract_digits(num))          # [1, 2, 3, 4, 5]
print(extract_digits(-9876))        # [-9, 8, 7, 6]

[1, 2, 3, 4, 5]
[-9, 8, 7, 6]


In [5]:
from typing import List

class Solution:
    def minimum_cost(self, cost: List[int]) -> int:
        if not cost:
            return 0

        cost.sort(reverse=True)

        total = 0
        n = len(cost)

        for i in range(n):
            if i % 3 != 2:
                total += cost[i]

        return total

l1 = [2, 3, 4, 5, 6, 7]

obj = Solution()
print(obj.minimum_cost(l1))


20


In [1]:
from typing import List

class Solution:
    def earliestFinishTime(self,
                          landStartTime: List[int],
                          landDuration: List[int],
                          waterStartTime: List[int],
                          waterDuration: List[int]) -> int:

        n = len(landStartTime)
        m = len(waterStartTime)

        if n == 0 or m == 0:
            return -1  # invalid case, though constraints say n,m >=1

        min_finish = float('inf')

        # Try all possible combinations
        for i in range(n):
            for j in range(m):

                # Case 1: Land -> Water
                finish_land = landStartTime[i] + landDuration[i]
                start_water = max(finish_land, waterStartTime[j])
                finish1 = start_water + waterDuration[j]

                # Case 2: Water -> Land
                finish_water = waterStartTime[j] + waterDuration[j]
                start_land = max(finish_water, landStartTime[i])
                finish2 = start_land + landDuration[i]

                # Best for this pair
                best_for_pair = min(finish1, finish2)

                # Track global minimum
                min_finish = min(min_finish, best_for_pair)

        return min_finish

In [1]:
class Solution:
    def earliestFinishTime(self, landStartTime: List[int], landDuration: List[int], waterStartTime: List[int], waterDuration: List[int]) -> int:
        if not landStartTime or not waterStartTime:
            return -1  # though constraints ensure at least 1

        # Compute end times
        endL = [landStartTime[i] + landDuration[i] for i in range(len(landStartTime))]
        endW = [waterStartTime[j] + waterDuration[j] for j in range(len(waterStartTime))]

        min_endL = min(endL)
        min_endW = min(endW)

        # minA: land first, using best land
        minA = float('inf')
        for j in range(len(waterStartTime)):
            val = max(min_endL + waterDuration[j], endW[j])
            minA = min(minA, val)

        # minB: water first, using best water
        minB = float('inf')
        for i in range(len(landStartTime)):
            val = max(min_endW + landDuration[i], endL[i])
            minB = min(minB, val)

        return min(minA, minB)

Efficient approach:

Store all positions of each element in a dictionary.
For a query (l, r, x), find how many indices of x lie in [l, r].
Since positions are sorted, use binary search (bisect_left and bisect_right).

Time Complexity:

Preprocessing: O(n)
Each query: O(log n)
Total: O(n + q log n)

In [2]:
from bisect import bisect_left, bisect_right
from collections import defaultdict

class Solution:
    def freqInRange(self, arr, queries):
        pos = defaultdict(list)

        # Store indices of each value
        for i, val in enumerate(arr):
            pos[val].append(i)

        ans = []

        for l, r, x in queries:
            if x not in pos:
                ans.append(0)
                continue

            indices = pos[x]

            left = bisect_left(indices, l)
            right = bisect_right(indices, r)

            ans.append(right - left)

        return ans

In [1]:
from functools import lru_cache

class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:

        def solve(n: int) -> int:
            if n < 0:
                return 0

            digits = list(map(int, str(n)))
            m = len(digits)

            @lru_cache(None)
            def dp(pos, tight, started, length, prev2, prev1):
                if pos == m:
                    return (1, 0)  # (count, total_waviness)

                limit = digits[pos] if tight else 9

                total_count = 0
                total_waviness = 0

                for d in range(limit + 1):
                    ntight = tight and (d == limit)

                    if not started and d == 0:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            False,
                            0,
                            10,
                            10
                        )
                        total_count += cnt
                        total_waviness += wav

                    elif not started:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            1,
                            10,
                            d
                        )
                        total_count += cnt
                        total_waviness += wav

                    else:
                        if length == 1:
                            cnt, wav = dp(
                                pos + 1,
                                ntight,
                                True,
                                2,
                                prev1,
                                d
                            )
                            total_count += cnt
                            total_waviness += wav

                        else:
                            add = 0
                            if (prev1 > prev2 and prev1 > d) or \
                               (prev1 < prev2 and prev1 < d):
                                add = 1

                            cnt, wav = dp(
                                pos + 1,
                                ntight,
                                True,
                                2,
                                prev1,
                                d
                            )

                            total_count += cnt
                            total_waviness += wav + add * cnt

                return (total_count, total_waviness)

            return dp(0, True, False, 0, 10, 10)[1]

        return solve(num2) - solve(num1 - 1)

In [ ]:
from functools import lru_cache

class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:

        def solve(n: int) -> int:
            if n < 0:
                return 0

            digits = list(map(int, str(n)))
            m = len(digits)

            @lru_cache(None)
            def dp(pos, tight, started, length, prev2, prev1):
                if pos == m:
                    return (1, 0)  # (count, total_waviness)

                limit = digits[pos] if tight else 9

                total_cnt = 0
                total_wav = 0

                for d in range(limit + 1):
                    ntight = tight and (d == limit)

                    # Still leading zeros
                    if not started and d == 0:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            False,
                            0,
                            10,
                            10
                        )
                        total_cnt += cnt
                        total_wav += wav
                        continue

                    if not started:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            1,
                            10,
                            d
                        )
                        total_cnt += cnt
                        total_wav += wav
                    else:
                        contrib = 0

                        if length >= 2:
                            # prev1 is the middle digit
                            if (prev1 > prev2 and prev1 > d) or \
                               (prev1 < prev2 and prev1 < d):
                                contrib = 1

                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            length + 1,
                            prev1,
                            d
                        )

                        total_cnt += cnt
                        total_wav += wav + contrib * cnt

                return (total_cnt, total_wav)

            return dp(0, True, False, 0, 10, 10)[1]

        return solve(num2) - solve(num1 - 1)

In [1]:
class Solution:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        n = len(nums)
        if n == 0:
            return []

        total = sum(nums)
        left_sum = 0
        answer = []

        for i in range(n):
            right_sum = total - left_sum - nums[i]
            answer.append(abs(left_sum - right_sum))
            left_sum += nums[i]

        return answer

In [1]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MAX = 2**31 - 1
        INT_MIN = -2**31

        sign = -1 if x < 0 else 1
        x = abs(x)

        rev = 0

        while x:
            digit = x % 10
            x //= 10

            if rev > INT_MAX // 10 or (rev == INT_MAX // 10 and digit > 7):
                return 0

            rev = rev * 10 + digit

        rev *= sign

        return rev if INT_MIN <= rev <= INT_MAX else 0

In [ ]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def createBinaryTree(self, descriptions: List[List[int]]) -> Optional[TreeNode]:
        nodes = {}
        children = set()

        for parent, child, isLeft in descriptions:
            if parent not in nodes:
                nodes[parent] = TreeNode(parent)

            if child not in nodes:
                nodes[child] = TreeNode(child)

            if isLeft:
                nodes[parent].left = nodes[child]
            else:
                nodes[parent].right = nodes[child]

            children.add(child)

        for val in nodes:
            if val not in children:
                return nodes[val]
        return None

In [1]:
"""
Sliding Window Pattern - Complete Implementation
All common sliding window problems with function-based interface
"""

from typing import List, Dict, Optional
from collections import defaultdict, Counter

class SlidingWindow:
    """Complete sliding window patterns collection"""

    @staticmethod
    def max_sum_subarray_of_size_k(arr: List[int], k: int) -> int:
        """
        Problem 1: Maximum Sum Subarray of Size K
        Time: O(n), Space: O(1)
        """
        if not arr or k > len(arr):
            return -1

        window_sum = sum(arr[:k])
        max_sum = window_sum

        for i in range(k, len(arr)):
            window_sum = window_sum - arr[i - k] + arr[i]
            max_sum = max(max_sum, window_sum)

        return max_sum

    @staticmethod
    def smallest_subarray_with_sum_given(arr: List[int], target: int) -> int:
        """
        Problem 2: Smallest Subarray with Sum >= Target
        Time: O(n), Space: O(1)
        """
        window_start = 0
        window_sum = 0
        min_length = float('inf')

        for window_end in range(len(arr)):
            window_sum += arr[window_end]

            while window_sum >= target:
                min_length = min(min_length, window_end - window_start + 1)
                window_sum -= arr[window_start]
                window_start += 1

        return min_length if min_length != float('inf') else 0

    @staticmethod
    def longest_substring_with_k_distinct(s: str, k: int) -> int:
        """
        Problem 3: Longest Substring with K Distinct Characters
        Time: O(n), Space: O(k)
        """
        if k == 0 or not s:
            return 0

        window_start = 0
        char_freq = {}
        max_length = 0

        for window_end in range(len(s)):
            right_char = s[window_end]
            char_freq[right_char] = char_freq.get(right_char, 0) + 1

            while len(char_freq) > k:
                left_char = s[window_start]
                char_freq[left_char] -= 1
                if char_freq[left_char] == 0:
                    del char_freq[left_char]
                window_start += 1

            max_length = max(max_length, window_end - window_start + 1)

        return max_length

    @staticmethod
    def longest_substring_without_repeating(s: str) -> int:
        """
        Problem 4: Longest Substring Without Repeating Characters
        Time: O(n), Space: O(min(n, alphabet))
        """
        window_start = 0
        char_index = {}
        max_length = 0

        for window_end in range(len(s)):
            if s[window_end] in char_index and char_index[s[window_end]] >= window_start:
                window_start = char_index[s[window_end]] + 1

            char_index[s[window_end]] = window_end
            max_length = max(max_length, window_end - window_start + 1)

        return max_length

    @staticmethod
    def character_replacement(s: str, k: int) -> int:
        """
        Problem 5: Longest Repeating Character Replacement
        Time: O(n), Space: O(26)
        """
        window_start = 0
        char_freq = {}
        max_count = 0
        max_length = 0

        for window_end in range(len(s)):
            char_freq[s[window_end]] = char_freq.get(s[window_end], 0) + 1
            max_count = max(max_count, char_freq[s[window_end]])

            while (window_end - window_start + 1) - max_count > k:
                char_freq[s[window_start]] -= 1
                window_start += 1

            max_length = max(max_length, window_end - window_start + 1)

        return max_length

    @staticmethod
    def max_consecutive_ones(nums: List[int], k: int) -> int:
        """
        Problem 6: Max Consecutive Ones After Flipping K Zeros
        Time: O(n), Space: O(1)
        """
        window_start = 0
        max_ones = 0
        zeros_count = 0

        for window_end in range(len(nums)):
            if nums[window_end] == 0:
                zeros_count += 1

            while zeros_count > k:
                if nums[window_start] == 0:
                    zeros_count -= 1
                window_start += 1

            max_ones = max(max_ones, window_end - window_start + 1)

        return max_ones

    @staticmethod
    def find_anagrams(s: str, p: str) -> List[int]:
        """
        Problem 7: Find All Anagrams in a String
        Time: O(n), Space: O(1)
        """
        if len(p) > len(s):
            return []

        p_count = [0] * 26
        s_count = [0] * 26
        result = []

        for char in p:
            p_count[ord(char) - ord('a')] += 1

        for i in range(len(s)):
            s_count[ord(s[i]) - ord('a')] += 1

            if i >= len(p):
                s_count[ord(s[i - len(p)]) - ord('a')] -= 1

            if s_count == p_count:
                result.append(i - len(p) + 1)

        return result

    @staticmethod
    def max_vowels_in_substring(s: str, k: int) -> int:
        """
        Problem 8: Maximum Number of Vowels in a Substring of Length K
        Time: O(n), Space: O(1)
        """
        vowels = set('aeiou')
        window_start = 0
        max_vowels = 0
        current_vowels = 0

        for window_end in range(len(s)):
            if s[window_end] in vowels:
                current_vowels += 1

            if window_end - window_start + 1 > k:
                if s[window_start] in vowels:
                    current_vowels -= 1
                window_start += 1

            if window_end - window_start + 1 == k:
                max_vowels = max(max_vowels, current_vowels)

        return max_vowels

    @staticmethod
    def subarray_product_less_than_k(nums: List[int], k: int) -> int:
        """
        Problem 9: Subarray Product Less Than K
        Time: O(n), Space: O(1)
        """
        if k <= 1:
            return 0

        count = 0
        product = 1
        window_start = 0

        for window_end in range(len(nums)):
            product *= nums[window_end]

            while product >= k and window_start <= window_end:
                product //= nums[window_start]
                window_start += 1

            count += window_end - window_start + 1

        return count

    @staticmethod
    def longest_subarray_with_ones_after_deleting(nums: List[int]) -> int:
        """
        Problem 10: Longest Subarray of 1's After Deleting One Element
        Time: O(n), Space: O(1)
        """
        window_start = 0
        zeros_count = 0
        max_length = 0

        for window_end in range(len(nums)):
            if nums[window_end] == 0:
                zeros_count += 1

            while zeros_count > 1:
                if nums[window_start] == 0:
                    zeros_count -= 1
                window_start += 1

            max_length = max(max_length, window_end - window_start + 1)

        return max_length - 1 if max_length > 0 else 0

    @staticmethod
    def min_window_substring(s: str, t: str) -> str:
        """
        Problem 11: Minimum Window Substring
        Time: O(n), Space: O(m) where m = len(t)
        """
        if not s or not t:
            return ""

        required = Counter(t)
        formed = 0
        window_counts = defaultdict(int)

        left = 0
        min_len = float('inf')
        min_left = 0

        for right in range(len(s)):
            char = s[right]
            window_counts[char] += 1

            if char in required and window_counts[char] == required[char]:
                formed += 1

            while formed == len(required):
                if right - left + 1 < min_len:
                    min_len = right - left + 1
                    min_left = left

                left_char = s[left]
                window_counts[left_char] -= 1
                if left_char in required and window_counts[left_char] < required[left_char]:
                    formed -= 1
                left += 1

        return s[min_left:min_left + min_len] if min_len != float('inf') else ""

    @staticmethod
    def contains_nearby_duplicate(nums: List[int], k: int) -> bool:
        """
        Problem 12: Contains Duplicate II (within K distance)
        Time: O(n), Space: O(min(n, k))
        """
        window = set()

        for i in range(len(nums)):
            if i > k:
                window.remove(nums[i - k - 1])

            if nums[i] in window:
                return True

            window.add(nums[i])

        return False

    @staticmethod
    def max_sum_subarray_size_k_sliding(arr: List[int], k: int) -> List[int]:
        """
        Problem 13: Return the subarray with maximum sum of size K
        Time: O(n), Space: O(k)
        """
        if not arr or k > len(arr):
            return []

        window_sum = sum(arr[:k])
        max_sum = window_sum
        max_start_index = 0

        for i in range(k, len(arr)):
            window_sum = window_sum - arr[i - k] + arr[i]
            if window_sum > max_sum:
                max_sum = window_sum
                max_start_index = i - k + 1

        return arr[max_start_index:max_start_index + k]

    @staticmethod
    def longest_turbulent_subarray(arr: List[int]) -> int:
        """
        Problem 14: Longest Turbulent Subarray
        Time: O(n), Space: O(1)
        """
        if len(arr) <= 1:
            return len(arr)

        max_len = 1
        window_start = 0

        for window_end in range(1, len(arr)):
            compare = (arr[window_end - 1] - arr[window_end])

            if compare == 0:
                window_start = window_end
            elif window_end == len(arr) - 1 or (arr[window_end] - arr[window_end + 1]) * compare >= 0:
                max_len = max(max_len, window_end - window_start + 1)
                window_start = window_end

        return max_len


# Function-based interface for easy calling
def sliding_window_examples():
    """Demonstrate all sliding window functions with examples"""

    sw = SlidingWindow()

    print("=" * 80)
    print("SLIDING WINDOW PATTERN - COMPLETE EXAMPLES")
    print("=" * 80)

    # Example 1: Maximum Sum Subarray of Size K
    print("\n1. Maximum Sum Subarray of Size K")
    arr1 = [2, 1, 5, 1, 3, 2]
    k1 = 3
    result1 = sw.max_sum_subarray_of_size_k(arr1, k1)
    print(f"   Array: {arr1}, K={k1}")
    print(f"   Result: {result1}")

    # Example 2: Smallest Subarray with Sum >= Target
    print("\n2. Smallest Subarray with Sum >= Target")
    arr2 = [2, 1, 5, 2, 3, 2]
    target2 = 7
    result2 = sw.smallest_subarray_with_sum_given(arr2, target2)
    print(f"   Array: {arr2}, Target={target2}")
    print(f"   Result: {result2}")

    # Example 3: Longest Substring with K Distinct Characters
    print("\n3. Longest Substring with K Distinct Characters")
    s3 = "araaci"
    k3 = 2
    result3 = sw.longest_substring_with_k_distinct(s3, k3)
    print(f"   String: '{s3}', K={k3}")
    print(f"   Result: {result3}")

    # Example 4: Longest Substring Without Repeating Characters
    print("\n4. Longest Substring Without Repeating Characters")
    s4 = "abcabcbb"
    result4 = sw.longest_substring_without_repeating(s4)
    print(f"   String: '{s4}'")
    print(f"   Result: {result4}")

    # Example 5: Character Replacement
    print("\n5. Longest Repeating Character Replacement")
    s5 = "AABABBA"
    k5 = 1
    result5 = sw.character_replacement(s5, k5)
    print(f"   String: '{s5}', K={k5}")
    print(f"   Result: {result5}")

    # Example 6: Max Consecutive Ones
    print("\n6. Max Consecutive Ones After Flipping K Zeros")
    nums6 = [1, 1, 0, 1, 1, 0, 1, 1, 1]
    k6 = 2
    result6 = sw.max_consecutive_ones(nums6, k6)
    print(f"   Array: {nums6}, K={k6}")
    print(f"   Result: {result6}")

    # Example 7: Find All Anagrams
    print("\n7. Find All Anagrams in a String")
    s7 = "cbaebabacd"
    p7 = "abc"
    result7 = sw.find_anagrams(s7, p7)
    print(f"   String: '{s7}', Pattern: '{p7}'")
    print(f"   Result: {result7}")

    # Example 8: Maximum Vowels in Substring
    print("\n8. Maximum Vowels in Substring of Length K")
    s8 = "abciiidef"
    k8 = 3
    result8 = sw.max_vowels_in_substring(s8, k8)
    print(f"   String: '{s8}', K={k8}")
    print(f"   Result: {result8}")

    # Example 9: Subarray Product Less Than K
    print("\n9. Subarray Product Less Than K")
    nums9 = [10, 5, 2, 6]
    k9 = 100
    result9 = sw.subarray_product_less_than_k(nums9, k9)
    print(f"   Array: {nums9}, K={k9}")
    print(f"   Result: {result9}")

    # Example 10: Longest Subarray of 1's After Deleting One Element
    print("\n10. Longest Subarray of 1's After Deleting One Element")
    nums10 = [1, 1, 0, 1, 1, 1, 0, 1, 1, 1]
    result10 = sw.longest_subarray_with_ones_after_deleting(nums10)
    print(f"   Array: {nums10}")
    print(f"   Result: {result10}")

    # Example 11: Minimum Window Substring
    print("\n11. Minimum Window Substring")
    s11 = "ADOBECODEBANC"
    t11 = "ABC"
    result11 = sw.min_window_substring(s11, t11)
    print(f"   String: '{s11}', Target: '{t11}'")
    print(f"   Result: '{result11}'")

    # Example 12: Contains Duplicate II
    print("\n12. Contains Duplicate II (within K distance)")
    nums12 = [1, 2, 3, 1, 2, 3]
    k12 = 2
    result12 = sw.contains_nearby_duplicate(nums12, k12)
    print(f"   Array: {nums12}, K={k12}")
    print(f"   Result: {result12}")

    # Example 13: Return Max Sum Subarray
    print("\n13. Return Subarray with Maximum Sum of Size K")
    arr13 = [2, 1, 5, 1, 3, 2]
    k13 = 3
    result13 = sw.max_sum_subarray_size_k_sliding(arr13, k13)
    print(f"   Array: {arr13}, K={k13}")
    print(f"   Result: {result13}")

    # Example 14: Longest Turbulent Subarray
    print("\n14. Longest Turbulent Subarray")
    arr14 = [9, 4, 2, 10, 7, 8, 8, 1, 9]
    result14 = sw.longest_turbulent_subarray(arr14)
    print(f"   Array: {arr14}")
    print(f"   Result: {result14}")

    print("\n" + "=" * 80)
    print("All sliding window patterns demonstrated successfully!")
    print("=" * 80)


# Individual function calls - easy to use
def get_max_sum_subarray(arr, k):
    """Wrapper function for max sum subarray"""
    return SlidingWindow.max_sum_subarray_of_size_k(arr, k)

def get_smallest_subarray_sum(arr, target):
    """Wrapper for smallest subarray with sum >= target"""
    return SlidingWindow.smallest_subarray_with_sum_given(arr, target)

def get_longest_k_distinct(s, k):
    """Wrapper for longest substring with k distinct characters"""
    return SlidingWindow.longest_substring_with_k_distinct(s, k)

def get_longest_unique_substring(s):
    """Wrapper for longest substring without repeating"""
    return SlidingWindow.longest_substring_without_repeating(s)

def get_max_consecutive_ones(nums, k):
    """Wrapper for max consecutive ones after flipping"""
    return SlidingWindow.max_consecutive_ones(nums, k)

def find_all_anagrams(s, p):
    """Wrapper for finding anagrams"""
    return SlidingWindow.find_anagrams(s, p)

def get_min_window_substring(s, t):
    """Wrapper for minimum window substring"""
    return SlidingWindow.min_window_substring(s, t)


# Main execution
if __name__ == "__main__":
    # Run all examples
    sliding_window_examples()

    # Or call individual functions:
    print("\n\nINDIVIDUAL FUNCTION CALLS:")
    print("-" * 50)

    # Example: Using individual functions
    arr = [1, 4, 2, 10, 23, 3, 1, 0, 20]
    k = 4
    result = get_max_sum_subarray(arr, k)
    print(f"Max sum of subarray of size {k}: {result}")

    s = "pwwkew"
    result = get_longest_unique_substring(s)
    print(f"Longest substring without repeating in '{s}': {result}")

    nums = [1, 1, 0, 1, 1, 1]
    k = 1
    result = get_max_consecutive_ones(nums, k)
    print(f"Max consecutive ones after flipping {k} zeros: {result}")

In [1]:
class Solution:
    def pivotArray(self, nums: List[int], pivot: int) -> List[int]:
        smaller = []
        equal = []
        larger = []
        for num in nums:
            if num < pivot:
                smaller.append(num)
            elif num == pivot:
                equal.append(num)
            else:
                larger.append(num)
        return smaller + equal + larger

In [2]:
class Solution:
    def pivotArray(self, nums: List[int], pivot: int) -> List[int]:
        # O(n) time, O(n) extra space (standard and efficient approach)
        # True O(1) extra space with stability + O(n) time is non-trivial/complex
        # (would require advanced techniques like stable in-place partitioning)
        smaller = []
        equal = []
        larger = []
        for num in nums:
            if num < pivot:
                smaller.append(num)
            elif num == pivot:
                equal.append(num)
            else:
                larger.append(num)
        return smaller + equal + larger

Aspect,"Dutch National Flag (e.g., Sort Colors)",This Pivot Array Problem
Stability,Unstable — relative order within each group is not preserved,Stable — relative order of all < pivot elements and all > pivot elements must be maintained
Extra Space,O(1) (classic in-place version),O(n) is straightforward; true O(1) extra space is complex
In-place,Yes (standard solution),Difficult while keeping stability
Number of distinct values,"Exactly 3 fixed values (0,1,2)","Arbitrary integers, pivot is one of them"
Algorithm Style,"Two-pointer / three-pointer technique (low, mid, high)",Usually two passes + temporary storage for stability
Use Case,Classic quicksort 3-way partition optimization,Stable reordering around a pivot

In [3]:
class Solution:
    def pivotArray(self, nums: List[int], pivot: int) -> List[int]:
        smaller = [x for x in nums if x < pivot]
        equal = [x for x in nums if x == pivot]
        larger = [x for x in nums if x > pivot]
        return smaller + equal + larger